# Who Gets Hospitalized with Flu?
### Analyzing 18 Seasons of CDC Influenza Hospitalization Data (2009–2026) by Age, Sex, Race and Virus Type

## Project Description
This project explores real influenza hospitalization data from the CDC's **Influenza Hospitalization Surveillance Network (FluSurv-NET)**. The network tracks lab-confirmed flu hospitalizations across selected counties in about 13 US states. The dataset covers **18 flu seasons (2009–10 to 2026–27)** and reports weekly and cumulative hospitalization rates per 100,000 people, broken down by **age group, sex, race/ethnicity, and virus type**.

**Workflow:**
1. **Data cleaning:** handle the raw CDC file format, missing values, and "Overall" summary rows.
2. **Exploratory data analysis:** compare hospitalization rates across age groups, sex, race/ethnicity, and virus types.
3. **Trend analysis:** compare seasons, find peak weeks, and spot unusually severe or mild years.
4. **Visualization:** use charts to show differences between groups and patterns over time.
5. **Insights and recommendations:** turn findings into practical suggestions for vaccination priorities and hospital preparation.

**Data source:** [CDC FluView Interactive – FluSurv-NET](https://gis.cdc.gov/GRASP/Fluview/FluHospRates.html)

> **Note:** The data is aggregated by group (not one row per patient) and covers selected areas, not the whole US.

---

## Objectives
1. **Identify high-risk groups:** find out which age groups, sexes and races/ethnicities have the highest flu hospitalization rates.
2. **Compare flu seasons:** measure how severe each season was (2009–10 to 2026–27) and find the worst and mildest seasons.
3. **Find seasonal patterns:** find when flu hospitalizations usually start rising, peak, and decline during the year.
4. **Compare virus types:** compare hospitalization rates caused by Influenza A vs. B, and by the A(H1N1) and A(H3N2) subtypes.
5. **Study the COVID-19 effect:** check how flu hospitalizations changed during the 2020–21 and 2021–22 seasons.
6. **Give recommendations:** turn the findings into practical suggestions for vaccination priorities and hospital planning.

---

## Dataset Features
**Size:** 28,758 rows × 16 columns

| Feature | Type | Description |
|---|---|---|
| `catchment` | Categorical | Area the data covers |
| `network` | Categorical | Surveillance network |
| `season` | Categorical | Flu season |
| `year` | Numeric (int) | Calendar year of the week |
| `week` | Numeric (int) | Week number of the year |
| `age_group` | Categorical | Age group |
| `sex` | Categorical | Sex |
| `race` | Categorical | Race/ethnicity |
| `virus_type` | Categorical | Flu virus type |
| `cumulative_rate` | Numeric (float) | Total hospitalizations per 100k so far in the season |
| `weekly_rate` | Numeric (float) | Hospitalizations per 100k in that week |
| `age_adjusted_cumulative_rate` | Numeric (float) | Cumulative rate adjusted for age |
| `age_adjusted_weekly_rate` | Numeric (float) | Weekly rate adjusted for age |
| `lower` | Numeric (float) | Lower nowcast estimate for recent weeks |
| `median` | Numeric (float) | Middle nowcast estimate for recent weeks |
| `upper` | Numeric (float) | Upper nowcast estimate for recent weeks |


In [52]:
#download data from drive
import gdown
file_id = "1x60gn8dXp3ULtVdznpxYOAkLGhM2J2Wb"
destination = "data.csv"   # name you want to save as

gdown.download(f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1x60gn8dXp3ULtVdznpxYOAkLGhM2J2Wb
To: /content/data.csv
100%|██████████| 3.25M/3.25M [00:00<00:00, 148MB/s]


'data.csv'

In [53]:
#import libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


#Read Data
original_df = pd.read_csv('data.csv')
df = original_df.copy()
df.head()

,CATCHMENT,NETWORK,YEAR,YEAR.1,WEEK,AGE CATEGORY,SEX CATEGORY,RACE CATEGORY,VIRUS TYPE CATEGORY,CUMULATIVE RATE,WEEKLY RATE,AGE ADJUSTED CUMULATIVE RATE,AGE ADJUSTED WEEKLY RATE,LOWER,MEDIAN,UPPER
0,Entire Network,FluSurv-NET,2009-10,2009,40,Overall,Overall,White,Overall,4.3,1.7,2.58,0.97,NaN,NaN,NaN
1,Entire Network,FluSurv-NET,2009-10,2009,40,Overall,Overall,Black,Overall,11.6,3.6,7.68,2.18,NaN,NaN,NaN
2,Entire Network,FluSurv-NET,2009-10,2009,40,Overall,Overall,Hispanic/Latino,Overall,12.8,4.8,7.50,2.73,NaN,NaN,NaN
3,Entire Network,FluSurv-NET,2009-10,2009,40,Overall,Overall,Asian/Pacific Islander,Overall,3.5,1.3,2.23,0.62,NaN,NaN,NaN
4,Entire Network,FluSurv-NET,2009-10,2009,40,Overall,Overall,American Indian/Alaska Native,Overall,10.4,4.3,5.52,0.40,NaN,NaN,NaN


In [54]:
df.tail()

,CATCHMENT,NETWORK,YEAR,YEAR.1,WEEK,AGE CATEGORY,SEX CATEGORY,RACE CATEGORY,VIRUS TYPE CATEGORY,CUMULATIVE RATE,WEEKLY RATE,AGE ADJUSTED CUMULATIVE RATE,AGE ADJUSTED WEEKLY RATE,LOWER,MEDIAN,UPPER
28753,Entire Network,FluSurv-NET,2026-27,2027,35,>= 18,Overall,Overall,Overall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28754,Entire Network,FluSurv-NET,2026-27,2027,36,>= 18,Overall,Overall,Overall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28755,Entire Network,FluSurv-NET,2026-27,2027,37,>= 18,Overall,Overall,Overall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28756,Entire Network,FluSurv-NET,2026-27,2027,38,>= 18,Overall,Overall,Overall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28757,Entire Network,FluSurv-NET,2026-27,2027,39,>= 18,Overall,Overall,Overall,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
df.shape

(28758, 16)

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28758 entries, 0 to 28757
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   CATCHMENT                     28758 non-null  object 
 1   NETWORK                       28758 non-null  object 
 2   YEAR                          28758 non-null  object 
 3   YEAR.1                        28758 non-null  int64  
 4   WEEK                          28758 non-null  int64  
 5   AGE CATEGORY                  28758 non-null  object 
 6   SEX CATEGORY                  28758 non-null  object 
 7   RACE CATEGORY                 28758 non-null  object 
 8   VIRUS TYPE CATEGORY           28758 non-null  object 
 9   CUMULATIVE RATE               25773 non-null  float64
 10  WEEKLY RATE                   25680 non-null  float64
 11  AGE ADJUSTED CUMULATIVE RATE  12840 non-null  float64
 12  AGE ADJUSTED WEEKLY RATE      12840 non-null  float64
 13  L

In [57]:
df.columns

Index(['CATCHMENT', 'NETWORK', 'YEAR', 'YEAR.1', 'WEEK', 'AGE CATEGORY',
       'SEX CATEGORY', 'RACE CATEGORY', 'VIRUS TYPE CATEGORY',
       'CUMULATIVE RATE', 'WEEKLY RATE', 'AGE ADJUSTED CUMULATIVE RATE',
       'AGE ADJUSTED WEEKLY RATE', 'LOWER', 'MEDIAN', 'UPPER'],
      dtype='object')

In [58]:
df.CATCHMENT.unique()

array(['Entire Network'], dtype=object)

In [59]:
#Note :CATCHMENT has one single Value we can Drop

In [60]:
df.drop(columns=["CATCHMENT"],inplace=True)

In [61]:
df.columns

Index(['NETWORK', 'YEAR', 'YEAR.1', 'WEEK', 'AGE CATEGORY', 'SEX CATEGORY',
       'RACE CATEGORY', 'VIRUS TYPE CATEGORY', 'CUMULATIVE RATE',
       'WEEKLY RATE', 'AGE ADJUSTED CUMULATIVE RATE',
       'AGE ADJUSTED WEEKLY RATE', 'LOWER', 'MEDIAN', 'UPPER'],
      dtype='object')

In [62]:
df.NETWORK.unique()

array(['FluSurv-NET'], dtype=object)

In [63]:
#Note : NETWORK has one single Value we can Drop

In [64]:
df.drop(columns=["NETWORK"],inplace=True)

In [65]:
df.columns

Index(['YEAR', 'YEAR.1', 'WEEK', 'AGE CATEGORY', 'SEX CATEGORY',
       'RACE CATEGORY', 'VIRUS TYPE CATEGORY', 'CUMULATIVE RATE',
       'WEEKLY RATE', 'AGE ADJUSTED CUMULATIVE RATE',
       'AGE ADJUSTED WEEKLY RATE', 'LOWER', 'MEDIAN', 'UPPER'],
      dtype='object')

In [66]:
df.YEAR.head()

,YEAR
0,2009-10
1,2009-10
2,2009-10
3,2009-10
4,2009-10


In [67]:
df["YEAR.1"].head()

,YEAR.1
0,2009
1,2009
2,2009
3,2009
4,2009


In [68]:
#Note : YEAR & YEAR.1 has the same Values so we can Drop one of it (YEAR.1)

In [69]:
df.drop(columns=["YEAR.1"],inplace=True)

In [70]:
df.columns

Index(['YEAR', 'WEEK', 'AGE CATEGORY', 'SEX CATEGORY', 'RACE CATEGORY',
       'VIRUS TYPE CATEGORY', 'CUMULATIVE RATE', 'WEEKLY RATE',
       'AGE ADJUSTED CUMULATIVE RATE', 'AGE ADJUSTED WEEKLY RATE', 'LOWER',
       'MEDIAN', 'UPPER'],
      dtype='object')

In [71]:
df.WEEK.unique()

array([40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52,  1,  2,  3,  4,
        5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 53, 18, 19, 20,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
       38, 39])

In [72]:
len(df.WEEK.unique())

53

In [73]:
df.columns

Index(['YEAR', 'WEEK', 'AGE CATEGORY', 'SEX CATEGORY', 'RACE CATEGORY',
       'VIRUS TYPE CATEGORY', 'CUMULATIVE RATE', 'WEEKLY RATE',
       'AGE ADJUSTED CUMULATIVE RATE', 'AGE ADJUSTED WEEKLY RATE', 'LOWER',
       'MEDIAN', 'UPPER'],
      dtype='object')

In [74]:
df["AGE CATEGORY"].unique()

array(['Overall', '0-4 yr', '5-17 yr', '18-49 yr', '50-64 yr', '>= 65 yr',
       '65-74 yr', '75-84 yr', '>= 85', '18-29 yr', '30-39 yr',
       '40-49 yr', '0-< 1 yr', '1-4 yr', '>= 75', '5-11  yr', '12-17 yr',
       '< 18', '>= 18'], dtype=object)

In [75]:
df.head(10)

,YEAR,WEEK,AGE CATEGORY,SEX CATEGORY,RACE CATEGORY,VIRUS TYPE CATEGORY,CUMULATIVE RATE,WEEKLY RATE,AGE ADJUSTED CUMULATIVE RATE,AGE ADJUSTED WEEKLY RATE,LOWER,MEDIAN,UPPER
0,2009-10,40,Overall,Overall,White,Overall,4.3,1.7,2.58,0.97,NaN,NaN,NaN
1,2009-10,40,Overall,Overall,Black,Overall,11.6,3.6,7.68,2.18,NaN,NaN,NaN
2,2009-10,40,Overall,Overall,Hispanic/Latino,Overall,12.8,4.8,7.50,2.73,NaN,NaN,NaN
3,2009-10,40,Overall,Overall,Asian/Pacific Islander,Overall,3.5,1.3,2.23,0.62,NaN,NaN,NaN
4,2009-10,40,Overall,Overall,American Indian/Alaska Native,Overall,10.4,4.3,5.52,0.40,NaN,NaN,NaN
5,2009-10,40,Overall,Male,Overall,Overall,6.4,2.6,NaN,NaN,NaN,NaN,NaN
6,2009-10,40,Overall,Female,Overall,Overall,8.3,3.0,NaN,NaN,NaN,NaN,NaN
7,2009-10,40,Overall,Overall,Overall,Influenza A,7.2,2.8,4.08,1.40,NaN,NaN,NaN
8,2009-10,40,Overall,Overall,Overall,Overall,7.4,2.8,4.08,1.40,NaN,NaN,NaN
9,2009-10,40,Overall,Overall,Overall,Influenza B,0.0,0.0,4.08,1.40,NaN,NaN,NaN


In [76]:
#NOTE:We noticed that AGE CATEGORY contains overlapping age groups and Overall rows, so we will select only the broad age groups to avoid overlapping comparisons.

In [77]:
age_groups = [
    '0-4 yr',
    '5-17 yr',
    '18-49 yr',
    '50-64 yr',
    '>= 65 yr'
]

df_age = df[
    df['AGE CATEGORY'].isin(age_groups) &
    (df['SEX CATEGORY'] == 'Overall') &
    (df['RACE CATEGORY'] == 'Overall') &
    (df['VIRUS TYPE CATEGORY'] == 'Overall')
].copy()

In [78]:
df_age.head(10)

,YEAR,WEEK,AGE CATEGORY,SEX CATEGORY,RACE CATEGORY,VIRUS TYPE CATEGORY,CUMULATIVE RATE,WEEKLY RATE,AGE ADJUSTED CUMULATIVE RATE,AGE ADJUSTED WEEKLY RATE,LOWER,MEDIAN,UPPER
302,2009-10,40,0-4 yr,Overall,Overall,Overall,18.1,7.2,4.08,1.40,NaN,NaN,NaN
303,2009-10,41,0-4 yr,Overall,Overall,Overall,28.3,10.2,6.47,2.40,NaN,NaN,NaN
308,2009-10,42,0-4 yr,Overall,Overall,Overall,39.1,10.8,9.58,3.10,NaN,NaN,NaN
309,2009-10,43,0-4 yr,Overall,Overall,Overall,47.3,8.2,13.08,3.50,NaN,NaN,NaN
314,2009-10,44,0-4 yr,Overall,Overall,Overall,53.3,6.0,16.18,3.10,NaN,NaN,NaN
315,2009-10,45,0-4 yr,Overall,Overall,Overall,57.5,4.2,18.60,2.42,NaN,NaN,NaN
320,2009-10,46,0-4 yr,Overall,Overall,Overall,60.1,2.6,20.18,1.58,NaN,NaN,NaN
321,2009-10,47,0-4 yr,Overall,Overall,Overall,61.6,1.5,21.33,1.15,NaN,NaN,NaN
326,2009-10,48,0-4 yr,Overall,Overall,Overall,62.9,1.3,22.01,0.68,NaN,NaN,NaN
327,2009-10,49,0-4 yr,Overall,Overall,Overall,64.1,1.3,22.46,0.45,NaN,NaN,NaN


In [79]:
df_age.shape

(3075, 13)